In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
import numpy as np
from tqdm import tqdm


In [ ]:
def sentiment_scores(sentence):
    sid_obj = SentimentIntensityAnalyzer()
    sentiment_dict = sid_obj.polarity_scores(sentence)

    print(f"Sentiment Scores: {sentiment_dict}")
    print(f"Negative: {sentiment_dict['neg']*100:.1f}%")
    print(f"Neutral: {sentiment_dict['neu']*100:.1f}%")
    print(f"Positive: {sentiment_dict['pos']*100:.1f}%")

    if sentiment_dict['compound'] >= 0.05:
        print("Overall: Positive")
    elif sentiment_dict['compound'] <= -0.05:
        print("Overall: Negative")
    else:
        print("Overall: Neutral")
    print("-" * 50)
    return sentiment_dict['compound']


In [ ]:
INPUT_CSV = 'earnings_calls.csv'
TEXT_COLUMN = 'transcript'

df = pd.read_csv(INPUT_CSV)
df = df.dropna(subset=[TEXT_COLUMN]).copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

# Keep only substantial transcripts and use a consistent column name for the analysis below.
df = df[df[TEXT_COLUMN].str.len() > 100].copy()
df = df.rename(columns={TEXT_COLUMN: 'transcript'})

df.head()


In [ ]:
# VADER
analyzer = SentimentIntensityAnalyzer()
compounds = []
negs, neus, poss = [], [], []

for text in tqdm(df['transcript'], desc="Analiza VADER"):
    scores = analyzer.polarity_scores(text)
    compounds.append(scores['compound'])
    negs.append(scores['neg'])
    neus.append(scores['neu'])
    poss.append(scores['pos'])

df['compound'] = compounds
df['neg'] = negs
df['neu'] = neus
df['pos'] = poss


In [ ]:
df['label'] = np.where(df['compound'] >= 0.05, 'POS',
               np.where(df['compound'] <= -0.05, 'NEG', 'NEU'))

print(df[['compound', 'label', 'neg', 'pos']].describe())
print("\nRozkład etykiet:")
print(df['label'].value_counts())

In [ ]:
OUTPUT_FILE = 'earnings_sentiment_vader.csv'

df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved {len(df):,} rows to {OUTPUT_FILE}')
